# LSTM vs SimpleRNN on a long-memory task
Real IMDB reviews: we keep the first 50 words of each review and then add a gap of padding steps after it.
The sentiment is in the words, so the model must carry it across the gap to the last time step.
Same data, same sizes, same training; only the recurrent layer (SimpleRNN or LSTM) and the gap change.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
from pathlib import Path
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

try:
    tf.config.experimental.enable_op_determinism()   # same numbers on every run
except Exception as e:
    print("op determinism not available:", e)
print("GPUs:", tf.config.list_physical_devices("GPU"))
DATA = Path("data")
DATA.mkdir(exist_ok=True)

## The data
5,000 training and 5,000 test reviews, 10,000 most frequent words. Each review is cut to its first 50 words
(shorter ones are padded in front), then `gap` padding steps (index 0) follow.

In [ ]:
VOCAB, KEEP, N = 10000, 50, 5000
(x_tr, y_tr), (x_te, y_te) = keras.datasets.imdb.load_data(num_words=VOCAB)
x_tr, x_te = x_tr[:N], x_te[:N]
y_tr, y_te = y_tr[:N].astype("float32"), y_te[:N].astype("float32")


def with_gap(x, gap):
    first = keras.utils.pad_sequences(x, maxlen=KEEP, padding="pre", truncating="post")   # first 50 words
    return np.concatenate([first, np.zeros((len(x), gap), dtype="int32")], axis=1).astype("int32")


print("shape with gap 25:", with_gap(x_tr, 25).shape, " positive share:", y_tr.mean())

## Training
Embedding (32 numbers per word), a recurrent layer of 32 units, a sigmoid output. Adam, 15 epochs, batch 64.
Every setting is trained with 10 seeds.

In [ ]:
GAPS, SEEDS, EPOCHS = (0, 25), tuple(range(10)), 15
rows = []
for gap in GAPS:
    xa, xb = with_gap(x_tr, gap), with_gap(x_te, gap)
    for kind in ("SimpleRNN", "LSTM"):
        for seed in SEEDS:
            keras.utils.set_random_seed(seed)
            model = keras.Sequential([keras.Input(shape=(KEEP + gap,)),
                                      keras.layers.Embedding(VOCAB, 32),
                                      getattr(keras.layers, kind)(32),
                                      keras.layers.Dense(1, activation="sigmoid")])
            model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
            h = model.fit(xa, y_tr, epochs=EPOCHS, batch_size=64, validation_data=(xb, y_te), verbose=0)
            for e, acc in enumerate(h.history["val_accuracy"], start=1):
                rows.append(dict(gap=gap, model=kind, seed=seed, epoch=e, val_accuracy=acc))
            print(f"gap {gap:3d} {kind:9} seed {seed}: final {h.history['val_accuracy'][-1]:.3f}", flush=True)
curves = pd.DataFrame(rows)
curves.round(4).to_csv(DATA / "val_accuracy.csv", index=False)

## Results: mean, spread and runs that learned, over 10 seeds

In [ ]:
final = curves[curves.epoch == EPOCHS].groupby(["gap", "model"]).val_accuracy
summary = final.agg(["mean", "std", "min", "max"]).round(3)
summary["learned (>0.6)"] = final.apply(lambda v: int((v > 0.6).sum()))
print(summary)
summary.to_csv(DATA / "summary.csv")
print("chance level (share of the larger class in the test set):", round(max(y_te.mean(), 1 - y_te.mean()), 3))

## A memory test with numbers: two series that differ only on day 1
Two shops report scaled daily sales for 4 days. Days 2 to 4 are identical; only day 1 differs (0 for shop A, 1 for shop B). The value to predict, day 5, equals day 1. A single LSTM unit must carry day 1 across three steps. Trained with 5 seeds on CPU; the long-term memory (cell state) and short-term memory (hidden state) of the first seed are saved day by day.

In [ ]:
seqs = np.array([[0.0, 0.6, 0.3, 0.9], [1.0, 0.6, 0.3, 0.9]], "float32")[:, :, None]   # shop A, shop B
day5 = np.array([0.0, 1.0], "float32")
demo_rows, demo_runs = [], []
for seed in range(5):
    keras.utils.set_random_seed(seed)
    lstm = keras.layers.LSTM(1, return_sequences=True, return_state=True)
    inp = keras.Input(shape=(None, 1))
    hs, h_last, c_last = lstm(inp)
    net = keras.Model(inp, h_last)                       # the last hidden state is the prediction for day 5
    net.compile(optimizer=keras.optimizers.Adam(0.05), loss="mse")
    hist = net.fit(seqs, day5, epochs=1500, batch_size=2, verbose=0).history
    pred = net.predict(seqs, verbose=0).ravel()
    demo_runs.append(dict(seed=seed, loss=hist["loss"][-1], pred_A=pred[0], pred_B=pred[1]))
    if seed == 0:
        states = keras.Model(inp, [h_last, c_last])
        for s, shop in enumerate("AB"):
            for day in range(1, 5):                      # states after reading days 1..day
                h, c = states.predict(seqs[s:s + 1, :day], verbose=0)
                demo_rows.append(dict(shop=shop, day=day, sales=float(seqs[s, day - 1, 0]),
                                      cell_state=float(c[0, 0]), hidden_state=float(h[0, 0])))
demo_runs = pd.DataFrame(demo_runs)
print(demo_runs.round(3))
pd.DataFrame(demo_rows).round(4).to_csv(DATA / "two_series_states.csv", index=False)
demo_runs.round(4).to_csv(DATA / "two_series_runs.csv", index=False)
print(pd.DataFrame(demo_rows).round(3))